# DroneDetect V2 - SVM Classification

Train SVM classifier on PSD features:
- RBF kernel (C=1.0, gamma='scale')
- File-level stratified split to prevent data leakage
- Comprehensive performance evaluation

## 0. Colab Setup

In [2]:
import os
import subprocess

COLAB = "COLAB_RELEASE_TAG" in os.environ

if COLAB:
    subprocess.check_call(["pip", "install", "-q", "boto3"])
    from google.colab import userdata
    import boto3

    s3 = boto3.client(
        "s3",
        endpoint_url="https://s3.fr-par.scw.cloud",
        region_name="fr-par",
        aws_access_key_id=userdata.get("SCW_ACCESS_KEY"),
        aws_secret_access_key=userdata.get("SCW_SECRET_KEY"),
    )
    ARTIFACT_BUCKET = "mldrone-artefacts"

    os.makedirs("../data/features", exist_ok=True)
    os.makedirs("../data/models", exist_ok=True)

    downloads = {
        "features/psd_features.npz": "../data/features/psd_features.npz",
        "split/split_indices.npz": "../data/split_indices.npz",
    }
    for s3_key, local_path in downloads.items():
        if not os.path.exists(local_path):
            print(f"Downloading {s3_key}...")
            s3.download_file(ARTIFACT_BUCKET, s3_key, local_path)
            print(f"Done: {local_path}")

Done: ../data/features/psd_features.npz
Done: ../data/split_indices.npz


## 1. Imports and Configuration

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, precision_recall_fscore_support
import pickle
import os
import re
from pathlib import Path

NOTEBOOK_NAME = "training_svm"
FIGURES_DIR = Path("figures") / NOTEBOOK_NAME


def save_figure(fig) -> None:
    """Save plotly figure to PNG file using the figure's title as filename."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    title = fig.layout.title.text if fig.layout.title.text else "untitled"
    filename = re.sub(r'[^\w\s-]', '', title).strip()
    filename = re.sub(r'[\s-]+', '_', filename)
    filepath = FIGURES_DIR / f"{filename}.png"
    try:
        fig.write_image(str(filepath), width=1200, height=800)
        print(f"Saved: {filepath}")
    except Exception as e:
        print(f"Warning: Could not save figure (kaleido required): {e}")

In [4]:
CONFIG = {
    'features_path': '../data/features/psd_features.npz',
    'models_dir': '../data/models/',
    'test_data_dir': '../data/sample/test_data/',
    'random_state': 42,
    'C': 1.0,
    'gamma': 'scale',
    'kernel': 'rbf'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  features_path: ../data/features/psd_features.npz
  models_dir: ../data/models/
  test_data_dir: ../data/sample/test_data/
  random_state: 42
  C: 1.0
  gamma: scale
  kernel: rbf


## 2. File-Level Stratified Split (dronedetect.splitting)

In [5]:
try:
    from dronedetect.splitting import create_stratified_split, verify_split_balance, load_split
except ImportError:
    import logging
    from pathlib import Path

    from sklearn.model_selection import StratifiedGroupKFold

    logger = logging.getLogger(__name__)

    def create_stratified_split(
        y: np.ndarray,
        file_ids: np.ndarray,
        train_size: float = 0.70,
        val_size: float = 0.15,
        test_size: float = 0.15,
        random_state: int = 42,
        save_path: str | Path | None = None,
    ) -> dict:
        """
        Two-stage stratified group split into train/val/test partitions.

        Stage 1: split off test set using StratifiedGroupKFold.
        Stage 2: split remaining trainval into train and val.

        Stratification is by drone label (y). Grouping is by file_ids to prevent
        segments from the same recording appearing in different partitions.

        Args:
            y: Drone labels array, shape (n_samples,).
            file_ids: File identifier per sample, shape (n_samples,).
            train_size: Fraction of data for training.
            val_size: Fraction of data for validation.
            test_size: Fraction of data for testing.
            random_state: Seed for reproducibility.
            save_path: If provided, save split indices to this .npz path.

        Returns:
            Dict with keys: train_idx, val_idx, test_idx, split_metadata.

        Raises:
            ValueError: If size fractions do not sum to 1 or leakage is detected.
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)

        total = train_size + val_size + test_size
        if not np.isclose(total, 1.0):
            raise ValueError(f"Split fractions must sum to 1.0, got {total:.4f}")

        n_splits_test = round(1 / test_size)
        sgkf_test = StratifiedGroupKFold(
            n_splits=n_splits_test, shuffle=True, random_state=random_state
        )
        all_indices = np.arange(len(y))

        trainval_idx, test_idx = next(sgkf_test.split(all_indices, y, groups=file_ids))

        relative_val = val_size / (train_size + val_size)
        n_splits_val = round(1 / relative_val)
        sgkf_val = StratifiedGroupKFold(
            n_splits=n_splits_val, shuffle=True, random_state=random_state + 1
        )

        y_trainval = y[trainval_idx]
        file_ids_trainval = file_ids[trainval_idx]

        train_local, val_local = next(
            sgkf_val.split(
                np.arange(len(trainval_idx)), y_trainval, groups=file_ids_trainval
            )
        )
        train_idx = trainval_idx[train_local]
        val_idx = trainval_idx[val_local]

        train_files = set(file_ids[train_idx])
        val_files = set(file_ids[val_idx])
        test_files = set(file_ids[test_idx])

        overlap_tv = train_files & val_files
        overlap_tt = train_files & test_files
        overlap_vt = val_files & test_files

        if overlap_tv or overlap_tt or overlap_vt:
            msg = "Data leakage detected! File overlap between partitions:"
            if overlap_tv:
                msg += f"\n  train-val: {overlap_tv}"
            if overlap_tt:
                msg += f"\n  train-test: {overlap_tt}"
            if overlap_vt:
                msg += f"\n  val-test: {overlap_vt}"
            raise ValueError(msg)

        logger.info("Split created with zero file overlap (leakage-free)")
        logger.info(
            "Samples  -> train=%d, val=%d, test=%d (total=%d)",
            len(train_idx),
            len(val_idx),
            len(test_idx),
            len(y),
        )
        logger.info(
            "Files    -> train=%d, val=%d, test=%d",
            len(train_files),
            len(val_files),
            len(test_files),
        )

        split_metadata = {
            "train_samples": len(train_idx),
            "val_samples": len(val_idx),
            "test_samples": len(test_idx),
            "train_files": len(train_files),
            "val_files": len(val_files),
            "test_files": len(test_files),
            "random_state": random_state,
        }

        result = {
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "split_metadata": split_metadata,
        }

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez(
                save_path,
                train_idx=train_idx,
                val_idx=val_idx,
                test_idx=test_idx,
            )
            logger.info("Split indices saved to %s", save_path)

        return result

    def verify_split_balance(
        file_ids: np.ndarray,
        y: np.ndarray,
        split_indices: dict,
        conditions: np.ndarray | None = None,
    ) -> None:
        """
        Log class and condition distributions per partition for diagnostics.

        Args:
            file_ids: File identifier per sample, shape (n_samples,).
            y: Drone labels array, shape (n_samples,).
            split_indices: Dict with train_idx, val_idx, test_idx.
            conditions: Optional condition labels (e.g. CLEAN/WIFI/BLUE/BOTH).
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)
        if conditions is not None:
            conditions = np.asarray(conditions)

        partitions = {
            "train": split_indices["train_idx"],
            "val": split_indices["val_idx"],
            "test": split_indices["test_idx"],
        }

        all_classes = np.unique(y)

        for name, idx in partitions.items():
            y_part = y[idx]
            classes, counts = np.unique(y_part, return_counts=True)
            dist_str = ", ".join(f"{c}={n}" for c, n in zip(classes, counts))
            logger.info("[%s] Class distribution: %s", name, dist_str)

        if conditions is not None:
            all_conditions = np.unique(conditions)

            for name, idx in partitions.items():
                cond_part = conditions[idx]
                conds, counts = np.unique(cond_part, return_counts=True)
                dist_str = ", ".join(f"{c}={n}" for c, n in zip(conds, counts))
                logger.info("[%s] Condition distribution: %s", name, dist_str)

                y_part = y[idx]
                for drone in all_classes:
                    drone_mask = y_part == drone
                    drone_conditions = set(np.unique(cond_part[drone_mask]))
                    missing = set(all_conditions) - drone_conditions
                    if missing:
                        logger.warning(
                            "[%s] Drone '%s' missing conditions: %s",
                            name,
                            drone,
                            missing,
                        )

    def load_split(path: str | Path) -> dict:
        """
        Load previously saved split indices from a .npz file.

        Args:
            path: Path to the .npz file created by create_stratified_split.

        Returns:
            Dict with keys: train_idx, val_idx, test_idx.
        """
        data = np.load(path)
        return {
            "train_idx": data["train_idx"],
            "val_idx": data["val_idx"],
            "test_idx": data["test_idx"],
        }

SPLIT_PATH = Path("../data/split_indices.npz")

print("Split module loaded")

Split module loaded


## 3. Load PSD Features

In [6]:
data = np.load(CONFIG['features_path'])

X = data['X']
y_drone = data['y_drone']
y_interference = data['y_interference']
y_state = data['y_state']
file_ids = data['file_ids']
drone_classes = data['drone_classes']
interference_classes = data['interference_classes']
state_classes = data['state_classes']

print(f"Features shape: {X.shape}")
print(f"Drone labels shape: {y_drone.shape}")
print(f"File IDs shape: {file_ids.shape} (unique files: {len(np.unique(file_ids))})")
print(f"Drone classes: {drone_classes}")
print(f"Interference classes: {interference_classes}")
print(f"State classes: {state_classes}")

Features shape: (39000, 1024)
Drone labels shape: (39000,)
File IDs shape: (39000,) (unique files: 390)
Drone classes: ['AIR' 'DIS' 'INS' 'MA1' 'MAV' 'MIN' 'PHA']
Interference classes: ['BLUE' 'BOTH' 'CLEAN' 'WIFI']
State classes: ['FY' 'HO' 'ON']


## 4. Train/Test Split

In [7]:
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
else:
    split = create_stratified_split(
        y=y_drone,
        file_ids=file_ids,
        save_path=SPLIT_PATH,
    )

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

# Val set not used by SVM (single fit), available for future hyperparameter tuning

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_drone[train_idx], y_drone[test_idx]
y_interference_test = y_interference[test_idx]
y_state_test = y_state[test_idx]

verify_split_balance(file_ids, y_drone, split, conditions=y_interference)

print(f"\nTraining set: {X_train.shape}")
print(f"Val set: {X[val_idx].shape}")
print(f"Test set: {X_test.shape}")

test_data_dir = CONFIG['test_data_dir']
os.makedirs(test_data_dir, exist_ok=True)

test_data_path = os.path.join(test_data_dir, 'svm_test_data.npz')
np.savez(
    test_data_path,
    X_test=X_test,
    y_test=y_test,
    y_interference_test=y_interference_test,
    y_state_test=y_state_test,
    test_idx=test_idx,
    file_ids_test=file_ids[test_idx],
    drone_classes=drone_classes,
    interference_classes=interference_classes,
    state_classes=state_classes
)
print(f"\nTest data saved to {test_data_path}")

print("\nGenerating separated test files (structure: psd/INT/DRONE/)...")

for d_idx, drone_class in enumerate(drone_classes):
    for i_idx, int_class in enumerate(interference_classes):
        mask = (y_test == d_idx) & (y_interference_test == i_idx)

        if not np.any(mask):
            continue

        X_sub = X_test[mask]
        y_sub = y_test[mask]
        y_int_sub = y_interference_test[mask]

        data_type = 'psd'
        int_name = str(int_class)
        drone_name = str(drone_class)
        duration = '20'

        save_dir = os.path.join(test_data_dir, int_name)
        os.makedirs(save_dir, exist_ok=True)

        filename = f"{data_type}_{int_name}_{drone_name}_{duration}.npz"
        file_path = os.path.join(save_dir, filename)

        np.savez(
            file_path,
            X=X_sub,
            y=y_sub,
            y_interference=y_int_sub,
            drone_class=drone_class,
            interference_class=int_class
        )
        print(f"  Saved {filename} in {save_dir}")


Training set: (27778, 1024)
Val set: (5600, 1024)
Test set: (5622, 1024)

Test data saved to ../data/sample/test_data/svm_test_data.npz

Generating separated test files (structure: psd/INT/DRONE/)...
  Saved psd_BLUE_AIR_20.npz in ../data/sample/test_data/BLUE
  Saved psd_BOTH_AIR_20.npz in ../data/sample/test_data/BOTH
  Saved psd_CLEAN_AIR_20.npz in ../data/sample/test_data/CLEAN
  Saved psd_WIFI_AIR_20.npz in ../data/sample/test_data/WIFI
  Saved psd_BLUE_DIS_20.npz in ../data/sample/test_data/BLUE
  Saved psd_BOTH_DIS_20.npz in ../data/sample/test_data/BOTH
  Saved psd_CLEAN_DIS_20.npz in ../data/sample/test_data/CLEAN
  Saved psd_WIFI_DIS_20.npz in ../data/sample/test_data/WIFI
  Saved psd_BLUE_INS_20.npz in ../data/sample/test_data/BLUE
  Saved psd_BOTH_INS_20.npz in ../data/sample/test_data/BOTH
  Saved psd_CLEAN_INS_20.npz in ../data/sample/test_data/CLEAN
  Saved psd_WIFI_INS_20.npz in ../data/sample/test_data/WIFI
  Saved psd_BLUE_MA1_20.npz in ../data/sample/test_data/BLUE


## 5. Train SVM Classifier

In [8]:
svm_model = SVC(C=CONFIG['C'], gamma=CONFIG['gamma'], kernel=CONFIG['kernel'])

print("Training SVM...")
svm_model.fit(X_train, y_train)
print("Training complete!")

Training SVM...
Training complete!


## 6. Evaluation

In [9]:
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1-Score (weighted): {f1:.4f}")

present_labels = sorted(set(y_test) | set(y_pred))
present_names = [drone_classes[i] for i in present_labels]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

Test Accuracy: 0.8346
Test F1-Score (weighted): 0.8304

Classification Report:
              precision    recall  f1-score   support

         AIR       0.67      0.78      0.72       922
         DIS       1.00      0.98      0.99       600
         INS       0.90      0.97      0.93       800
         MA1       0.81      0.54      0.65      1000
         MAV       0.70      0.75      0.72       800
         MIN       1.00      1.00      1.00       800
         PHA       0.87      0.95      0.91       700

    accuracy                           0.83      5622
   macro avg       0.85      0.85      0.85      5622
weighted avg       0.84      0.83      0.83      5622



## 7. Confusion Matrix

In [10]:
cm = confusion_matrix(y_test, y_pred)

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=list(drone_classes),
    y=list(drone_classes),
    colorscale='Blues',
    text=cm,
    texttemplate='%{text}',
    textfont={'size': 12},
    hoverongaps=False
))

fig.update_layout(
    title=f'SVM Confusion Matrix - Accuracy: {accuracy:.4f} | F1: {f1:.4f}',
    xaxis_title='Predicted',
    yaxis_title='True',
    xaxis={'side': 'bottom'},
    yaxis={'autorange': 'reversed'},
    width=800,
    height=700
)
fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 8. Per-Class Performance

In [11]:
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test, y_pred, labels=range(len(drone_classes)), zero_division=0
)

metrics_df = pd.DataFrame({
    'Class': drone_classes,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1_per_class,
    'Support': support
})

print("\nPer-Class Performance:")
print(metrics_df.to_string(index=False))

fig_precision = px.bar(metrics_df, x='Class', y='Precision', title='SVM Precision per Class',
                       color='Precision', range_y=[0, 1.05])
fig_precision.update_layout(xaxis_title="Class", yaxis_title="Precision",
                            title_font_size=16, height=400)
fig_precision.show()
save_figure(fig_precision)

fig_recall = px.bar(metrics_df, x='Class', y='Recall', title='SVM Recall per Class',
                    color='Recall', color_continuous_scale=px.colors.sequential.Oranges,
                    range_y=[0, 1.05])
fig_recall.update_layout(xaxis_title="Class", yaxis_title="Recall",
                         title_font_size=16, height=400)
fig_recall.show()
save_figure(fig_recall)

fig_f1 = px.bar(metrics_df, x='Class', y='F1-Score', title='SVM F1-Score per Class',
                color='F1-Score', color_continuous_scale=px.colors.sequential.Greens,
                range_y=[0, 1.05])
fig_f1.update_layout(xaxis_title="Class", yaxis_title="F1-Score",
                     title_font_size=16, height=400)
fig_f1.show()
save_figure(fig_f1)


Per-Class Performance:
Class  Precision   Recall  F1-Score  Support
  AIR   0.671642 0.780911  0.722166      922
  DIS   1.000000 0.978333  0.989048      600
  INS   0.896433 0.973750  0.933493      800
  MA1   0.806548 0.542000  0.648325     1000
  MAV   0.698026 0.751250  0.723660      800
  MIN   1.000000 1.000000  1.000000      800
  PHA   0.871222 0.947143  0.907598      700


Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 9. Save Model

In [12]:
os.makedirs(CONFIG['models_dir'], exist_ok=True)

model_path = os.path.join(CONFIG['models_dir'], 'svm_psd_drone.pkl')

with open(model_path, 'wb') as f:
    pickle.dump({
        'model': svm_model,
        'classes': drone_classes,
        'accuracy': accuracy,
        'f1': f1
    }, f)

print(f"Model saved to {model_path}")

Model saved to ../data/models/svm_psd_drone.pkl


## 10. Upload Model to Scaleway

In [13]:
if COLAB:
    model_file = os.path.join(CONFIG["models_dir"], "svm_psd_drone.pkl")
    s3.upload_file(model_file, ARTIFACT_BUCKET, "models/svm_psd_drone.pkl")
    print("Uploaded svm_psd_drone.pkl to Scaleway S3")

Uploaded svm_psd_drone.pkl to Scaleway S3


## 11. Summary

In [14]:
print("="*60)
print("SVM CLASSIFICATION SUMMARY")
print("="*60)

print("\nDataset:")
print(f"  Total samples: {len(X_train) + len(X[val_idx]) + len(X_test)}")
print(f"  Training samples: {len(X_train)}")
print(f"  Validation samples: {len(X[val_idx])} (unused by SVM)")
print(f"  Test samples: {len(X_test)}")
print(f"  Number of classes: {len(drone_classes)}")
print(f"  Classes: {', '.join(drone_classes)}")

print("\nModel Configuration:")
print("  Algorithm: Support Vector Machine (SVM)")
print(f"  Kernel: {CONFIG['kernel']}")
print(f"  C: {CONFIG['C']}")
print(f"  Gamma: {CONFIG['gamma']}")
print("  Features: PSD (Power Spectral Density)")

print("\nPerformance:")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  Test F1-Score: {f1:.4f}")

print(f"\nModel saved to: {model_path}")

print("\n" + "="*60)

SVM CLASSIFICATION SUMMARY

Dataset:
  Total samples: 39000
  Training samples: 27778
  Validation samples: 5600 (unused by SVM)
  Test samples: 5622
  Number of classes: 7
  Classes: AIR, DIS, INS, MA1, MAV, MIN, PHA

Model Configuration:
  Algorithm: Support Vector Machine (SVM)
  Kernel: rbf
  C: 1.0
  Gamma: scale
  Features: PSD (Power Spectral Density)

Performance:
  Test Accuracy: 0.8346
  Test F1-Score: 0.8304

Model saved to: ../data/models/svm_psd_drone.pkl

